# Middleware
### Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the
#### following:
https://reference.langchain.com/python/langchain/middleware
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### Sumarization middleware

##### Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older
##### context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_groq import ChatGroq

### MessageBased Sumarization

llm = ChatGroq(
    model="qwen/qwen3.6-27b"
)

agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [5]:
### Run with a thread id
config={"configurable":{"thread_id":"test-1"}}


In [6]:
from openai.types.responses import response
# Alternative test data

questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 2+5?",
    "What is 2*6?",
    "What is 4*6?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Message: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='09588e3a-d453-43b3-ba10-3c7183622e6a'), AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What is 2+2?"\n2.  **Identify Core Task:** This is a basic arithmetic question.\n3.  **Perform Calculation:** 2 + 2 = 4.\n4.  **Formulate Response:** Keep it direct and accurate. "2 + 2 equals 4."\n5.  **Self-Correction/Verification:** The math is correct. The response is straightforward. No additional context needed.\n6.  **Output Generation:** Provide the answer.✅\n</think>\n\n2 + 2 equals 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 137, 'prompt_tokens': 17, 'total_tokens': 154, 'completion_time': 0.267203603, 'completion_tokens_details': None, 'prompt_time': 0.001150539, 'prompt_tokens_details': None, 'queue_time': 0.046273066, 'total_time': 0.268354142}, 'model_name': 'qwen/qwen3.6-27b', 'system

# Token Size

In [9]:
from cryptography.x509 import certificate_transparency
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


llm = ChatGroq(
    model="qwen/qwen3.6-27b"
)

agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config={"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 char = 1 token



In [10]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:

        response = agent.invoke(
            {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
            config=config
        )

        tokens = count_tokens(response["messages"])
        print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
        print(f"{(response['messages'])}")

Paris: ~124 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='5dee1875-a3e2-4e21-8a2e-9a7842c1b6b7'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n\n1.  **Identify the User\'s Request:** The user wants to find hotels in "Paris".\n2.  **Identify the Available Tool:** There is a function `search_hotels` available.\n3.  **Analyze the Tool\'s Parameters:**\n    *   `city` (string, required): The city name.\n4.  **Map Request to Tool:**\n    *   User input: "Paris"\n    *   Tool parameter `city`: "Paris"\n5.  **Construct the Tool Call:** `search_hotels(city="Paris")`.\n6.  **Execute Tool Call:** (Simulated here, actual execution happens in the system).\n\n*Self-Correction/Refinement:* The prompt mentions the response is long and uses more tokens, likely implying I should just make the call and let the system handle the response or that I should be aware of the output size. No special action n

KeyboardInterrupt: 

In [ ]:
from cryptography.x509 import certificate_transparency
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


llm = ChatGroq(
    model="qwen/qwen3.6-27b"
)

agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction",0.005),
            keep=("fraction",0.002)
        )
    ]
)

config={"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 char = 1 token



# Human In the Loop MiddleWare
### Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the
### following:
* High-stakes operations requiring human approval (e.g. database writes, financial transactions).
* Compliance workflows where human oversight is mandatory.
* Long-running conversations where human feedback guides the agent.

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [12]:
llm = ChatGroq(
    model="qwen/qwen3.6-27b"
)

agent = create_agent(
    model=llm,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":{"approve","edit","reject"}
                },
                "read_email_tool":False
            }
        )
    ]
)

In [20]:
config = {"configurable": {"thread_id": "test:approve"}}
# Step 1: Request
result = agent.invoke(
                {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
                config=config
        )

In [14]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='e3fa467a-d426-46bc-afec-c1c995659e16'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email.\nI need to call the `send_email_tool` function.\nThe required parameters are `recipient`, `subject`, and `body`.\nFrom the user\'s request:\n- recipient: john@test.com\n- subject: Hello\n- body: How are you?\n\nI have all the required information to make the function call.\nI will construct the function call with these parameters.\nThen I will execute the function call.\nFinally, I will provide a response to the user confirming the action.\nNo other tools are needed.\nThe request is straightforward.\nProceeding with the function call. \nParameters:\nrecipient: "john@test.com"\nsubject: "Hello"\nbody: "How are you?"\nAll match the schema.\nReady. \nOutput matches the expected format.\nDone. \n[S

In [21]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("Pauseed ! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"reject"
                    }
                ]
            }
        ),
        config=config
    )

    print(f"Result : { result['messages']}")

Pauseed ! Approving...
Result : [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='e3fa467a-d426-46bc-afec-c1c995659e16'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email.\nI need to call the `send_email_tool` function.\nThe required parameters are `recipient`, `subject`, and `body`.\nFrom the user\'s request:\n- recipient: john@test.com\n- subject: Hello\n- body: How are you?\n\nI have all the required information to make the function call.\nI will construct the function call with these parameters.\nThen I will execute the function call.\nFinally, I will provide a response to the user confirming the action.\nNo other tools are needed.\nThe request is straightforward.\nProceeding with the function call. \nParameters:\nrecipient: "john@test.com"\nsubject: "Hello"\nbody: "How are you?"\nAll match the schema.\nReady. \nOutput matches the expected fo

In [22]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='e3fa467a-d426-46bc-afec-c1c995659e16'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email.\nI need to call the `send_email_tool` function.\nThe required parameters are `recipient`, `subject`, and `body`.\nFrom the user\'s request:\n- recipient: john@test.com\n- subject: Hello\n- body: How are you?\n\nI have all the required information to make the function call.\nI will construct the function call with these parameters.\nThen I will execute the function call.\nFinally, I will provide a response to the user confirming the action.\nNo other tools are needed.\nThe request is straightforward.\nProceeding with the function call. \nParameters:\nrecipient: "john@test.com"\nsubject: "Hello"\nbody: "How are you?"\nAll match the schema.\nReady. \nOutput matches the expected format.\nDone. \n[S